# Phase 4 — Submission 06: Model 3 Statutory Precision Full-MHA (Knowledge Distillation Student)

**Task:** Court Judgment Prediction (CJPE) in Indian Legal Documents (ILDC Benchmark)  
**Task Type:** Document-Level Binary Classification (`0` = Dismissed/Rejected, `1` = Allowed/Accepted)  
**Primary Evaluation Metric:** Macro-F1 Score  
**Student Backbone:** Phase 1 Model 3 Statutory Precision Full-MHA Decoder (~109.5M parameters, $L=12, d=768, H=12$, SwiGLU $d_{ff}=2048$, RoPE)  
**Teacher Backbone:** Phase 1 Model 4 Peak Capacity Boundary Decoder (~153.9M parameters, $L=14, d=896$, Sub 01 Anchor)  
**Pretrained Hub Location:** `sarweshbarnwal/CS781-Legal-Models` (subfolder: `model3-statutory-mha-110m`)  
**Target Hub Destination:** `sarweshbarnwal/CS781-Legal-Models` (subfolder: `Phase_4/sub06_model3_distill/`)  

---

### Strategic Architectural Role & Capacity Reduction
This notebook fulfills the **Compulsory Knowledge Distillation Requirement** with a genuine, validated capacity reduction:
1. **Teacher-Student Capacity Reduction:** Pairs the largest capacity teacher (**Model 4**, 153.9M parameters, $d=896$) with a compact student (**Model 3**, 109.5M parameters, $d=768$), achieving an empirical **28.8% capacity reduction** ($153.9\text{M} \to 109.5\text{M}$).
2. **Teacher Soft Logit Ingestion:** Ingests pre-computed teacher soft targets `sub01_teacher_logits.pt` exported by Sub 01 with automatic fallback to download from `Phase_4/sub01_model4_full_ft` on the Hugging Face Hub.
3. **Combined Distillation Objective:** Optimizes the student using both ground-truth hard labels and teacher soft probabilities:
   $$\mathcal{L} = 0.4 \cdot \mathcal{L}_{\text{BCE}}(y, z_{\text{student}}) + 0.6 \cdot T^2 \cdot \text{MSE}\left(\sigma(z_{\text{student}} / T), \, p_{\text{teacher}}\right)$$
   with temperature scaling $T = 2.0$ ($T^2 = 4.0$).
4. **Tail-Heavy Multi-Chunk Attention Pooling:** Documents are segmented into non-overlapping 1024-token chunks. The last $K=4$ chunks are pooled with learned attention over $[\text{MeanPool}(\mathbf{H}) \,;\, \mathbf{h}_{\text{last}}] \in \mathbb{R}^{1536}$.
5. **Robust Validation Pipeline:** Safely handles training batches with attached teacher targets while keeping validation and test routines purely focused on binary task evaluation without missing-target assertion errors.
6. **Self-Preservation & Hub Synchronization:** Preserves and stages the executed notebook itself into `checkpoint_sub06/` and atomically pushes all artifacts (`student_model_weights.safetensors`, `head.pt`, `attn_pool.pt`, `sub06_test_probs.npy`, `dev_probs.npy`, `val_metrics.json`, `loss_plot.png`, `submission.zip`, and `sub06_model3_distill.ipynb`) to the Hugging Face Hub.


In [1]:
# Cell 1: Imports, Environment Setup & Upfront Hugging Face Authentication
import os
import sys
import json
import math
import time
import shutil
import zipfile
import random
import getpass
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import PreTrainedTokenizerFast, AutoConfig, AutoModel, get_cosine_schedule_with_warmup
from safetensors.torch import load_file as safetensors_load_file, save_file as safetensors_save_file
from huggingface_hub import hf_hub_download, login, get_token, HfApi

# 1. Reproducible Seed Initialization
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 2. Compute Device Configuration with Architecture Compatibility Guard
def get_safe_device(preferred_device: Optional[str] = None) -> torch.device:
    if preferred_device:
        return torch.device(preferred_device)
    if torch.cuda.is_available():
        try:
            t = torch.zeros(1, device="cuda")
            _ = t + 1
            return torch.device("cuda")
        except Exception as e:
            print(f"Notice: CUDA is present but incompatible with current PyTorch build ({e}). Falling back to CPU.")
            return torch.device("cpu")
    return torch.device("cpu")

device = get_safe_device()
print(f"Active Compute Device: {device}")
if device.type == "cuda":
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

# 3. Repository and Target Configuration
MODEL_REPO = "sarweshbarnwal/CS781-Legal-Models"
TARGET_SUBFOLDER = "Phase_4/sub06_model3_distill"
TEACHER_SUBFOLDER = "Phase_4/sub01_model4_full_ft"
DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
BACKBONE_SUBFOLDER = "model3-statutory-mha-110m"
OUTPUT_DIR = "checkpoint_sub06"

# 4. Upfront Hugging Face Authentication
def resolve_hf_token(token: Optional[str] = None) -> Optional[str]:
    if token and str(token).strip():
        resolved = str(token).strip()
        try:
            login(token=resolved, add_to_git_credential=False)
        except Exception:
            pass
        return resolved

    for env_var in ["HF_TOKEN", "HF", "HUGGINGFACE_TOKEN"]:
        val = os.environ.get(env_var)
        if val and val.strip():
            resolved = val.strip()
            try:
                login(token=resolved, add_to_git_credential=False)
            except Exception:
                pass
            return resolved

    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        for sec_name in ["HF_TOKEN", "HF", "HUGGINGFACE_TOKEN"]:
            try:
                val = user_secrets.get_secret(sec_name)
                if val and val.strip():
                    resolved = val.strip()
                    try:
                        login(token=resolved, add_to_git_credential=False)
                    except Exception:
                        pass
                    return resolved
            except Exception:
                pass
    except Exception:
        pass

    try:
        hub_tok = get_token()
        if hub_tok and hub_tok.strip():
            return hub_tok.strip()
    except Exception:
        pass

    return None

HF_TOKEN = resolve_hf_token()
if HF_TOKEN:
    print(f"Hugging Face Authentication verified (token prefix: '{HF_TOKEN[:4]}****').")
else:
    print("Notice: HF_TOKEN not found in environment. You can enter it before upload.")


c:\Users\Sarwesh Barnwal\Desktop\ML\.venv_cs781\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1021 20:43:36.527000 24660 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Active Compute Device: cuda
GPU Device Name: NVIDIA GeForce RTX 5050 Laptop GPU
Hugging Face Authentication verified (token prefix: 'hf_r****').


In [2]:
# Cell 2: Universal Dataset Loader for Phase 4 (Court Judgment Prediction)
@dataclass
class LegalJudgmentDoc:
    doc_id: str
    text: str
    label: Optional[int] = None
    metadata: Dict[str, Any] = field(default_factory=dict)

    def __post_init__(self):
        if self.label is not None:
            self.label = int(self.label)
            if self.label not in (0, 1):
                raise ValueError(f"Document {self.doc_id} has invalid binary label: {self.label}")

def _normalize_dataset_records(raw_data: Any, split_name: str) -> List[LegalJudgmentDoc]:
    docs: List[LegalJudgmentDoc] = []
    if isinstance(raw_data, dict):
        for k, v in raw_data.items():
            if isinstance(v, dict):
                text = v.get("text") or v.get("document") or ""
                lbl = v.get("label") if "label" in v else v.get("labels", v.get("pred", None))
                docs.append(LegalJudgmentDoc(doc_id=str(k), text=str(text), label=lbl, metadata=v))
            elif isinstance(v, str):
                docs.append(LegalJudgmentDoc(doc_id=str(k), text=v, label=None))
    elif isinstance(raw_data, list):
        for idx, item in enumerate(raw_data):
            if isinstance(item, dict):
                doc_id = str(item.get("id") or item.get("doc_id") or f"{split_name}_{idx}")
                text = str(item.get("text") or item.get("document") or "")
                lbl = item.get("label") if "label" in item else item.get("labels", item.get("pred", None))
                docs.append(LegalJudgmentDoc(doc_id=doc_id, text=text, label=lbl, metadata=item))
            else:
                raise ValueError(f"Unrecognized item format at index {idx} in {split_name}")
    else:
        raise ValueError(f"Unrecognized dataset container type: {type(raw_data)} in {split_name}")

    if split_name in ("train", "dev"):
        for doc in docs:
            if doc.label is None:
                raise ValueError(f"FATAL: Missing label in {split_name} document: '{doc.doc_id}'")

    return docs

def load_phase4_splits(repo_id: str = DATASET_REPO, token: Optional[str] = None):
    splits_files = {
        "train": "Phase_4_CJPE/Train/train.json",
        "dev": "Phase_4_CJPE/Train/dev.json",
        "test": "Phase_4_CJPE/Test/test.json",
    }
    loaded = {}
    for key, path in splits_files.items():
        print(f"Downloading Phase 4 {key} split from '{repo_id}/{path}'...")
        local_path = hf_hub_download(repo_id=repo_id, filename=path, repo_type="dataset", token=token)
        with open(local_path, "r", encoding="utf-8") as f:
            raw_data = json.load(f)
        docs = _normalize_dataset_records(raw_data, key)
        loaded[key] = docs
        print(f"  -> Loaded {len(docs):,} {key} documents.")

    train_docs, dev_docs, test_docs = loaded["train"], loaded["dev"], loaded["test"]
    return train_docs, dev_docs, test_docs

train_docs, dev_docs, test_docs = load_phase4_splits(token=HF_TOKEN)
print(f"Dataset Loaded: {len(train_docs):,} Train, {len(dev_docs):,} Dev, {len(test_docs):,} Test")


  -> Loaded 37,387 train documents.
  -> Loaded 1,988 dev documents.
  -> Loaded 1,517 test documents.
Dataset Loaded: 37,387 Train, 1,988 Dev, 1,517 Test


In [3]:
# Cell 3: Teacher Soft Target Ingestion Pipeline (Sub 01 Model 4 Teacher)
# Ingests pre-computed teacher soft probabilities sub01_teacher_logits.pt exported by Sub 01
# Automatic fallback: searches local checkpoint directory, current working dir, and Hugging Face Hub

def load_teacher_soft_targets(
    repo_id: str = MODEL_REPO,
    teacher_subfolder: str = TEACHER_SUBFOLDER,
    token: Optional[str] = None
) -> Dict[str, float]:
    candidate_paths = [
        os.path.join("checkpoint_sub01", "sub01_teacher_logits.pt"),
        os.path.join("..", "checkpoint_sub01", "sub01_teacher_logits.pt"),
        "sub01_teacher_logits.pt",
        os.path.join("/kaggle/working", "checkpoint_sub01", "sub01_teacher_logits.pt"),
        os.path.join("/kaggle/working", "sub01_teacher_logits.pt"),
    ]

    for p in candidate_paths:
        if os.path.exists(p) and os.path.isfile(p):
            print(f"Found local teacher soft targets at: '{p}'")
            targets = torch.load(p, map_location="cpu")
            print(f"Successfully loaded {len(targets):,} teacher soft targets from local disk.")
            return targets

    # Fallback to download from Hugging Face Hub under Phase_4/sub01_model4_full_ft
    remote_filename = f"{teacher_subfolder}/sub01_teacher_logits.pt"
    print(f"Local file not found. Attempting download from Hugging Face Hub: '{repo_id}/{remote_filename}'...")
    try:
        hub_path = hf_hub_download(repo_id=repo_id, filename=remote_filename, token=token)
        targets = torch.load(hub_path, map_location="cpu")
        print(f"Successfully downloaded and loaded {len(targets):,} teacher soft targets from Hub.")
        return targets
    except Exception as e:
        print(f"Notice: Could not download teacher targets from Hub ({e}).")
        print("Falling back to temperature-calibrated pseudo-soft targets from ground-truth labels for smoke/dry runs.")
        pseudo_targets: Dict[str, float] = {}
        for d in train_docs:
            if d.label is not None:
                pseudo_targets[d.doc_id] = 0.85 if int(d.label) == 1 else 0.15
        return pseudo_targets

teacher_soft_targets = load_teacher_soft_targets(token=HF_TOKEN)
print(f"Teacher Soft Targets Ingested: {len(teacher_soft_targets):,} documents.")
sample_key = next(iter(teacher_soft_targets))
print(f"  Sample Target -> doc_id: '{sample_key}', soft_prob: {teacher_soft_targets[sample_key]:.4f}")


Local file not found. Attempting download from Hugging Face Hub: 'sarweshbarnwal/CS781-Legal-Models/Phase_4/sub01_model4_full_ft/sub01_teacher_logits.pt'...
Successfully downloaded and loaded 32,305 teacher soft targets from Hub.
Teacher Soft Targets Ingested: 32,305 documents.
  Sample Target -> doc_id: '2020_1', soft_prob: 0.5003


In [4]:
# Cell 4: Model 3 Tokenizer & Tail-Chunking Dataset Pipeline
def load_model3_tokenizer(repo_id: str = MODEL_REPO, token: Optional[str] = None) -> PreTrainedTokenizerFast:
    tok_path = hf_hub_download(repo_id=repo_id, filename=f"{BACKBONE_SUBFOLDER}/tokenizer.json", token=token)
    tok = PreTrainedTokenizerFast(
        tokenizer_file=tok_path,
        bos_token="<bos>",
        eos_token="<eos>",
        pad_token="<pad>",
        unk_token="<unk>"
    )
    tok.padding_side = "left"
    assert tok.pad_token_id == 3 and tok.eos_token_id == 1, f"Special token ID mismatch: pad={tok.pad_token_id}, eos={tok.eos_token_id}"
    return tok

tokenizer = load_model3_tokenizer(token=HF_TOKEN)

@dataclass
class PreprocessedDocumentBatch:
    doc_id: str
    num_chunks: int
    input_ids: torch.Tensor             # Shape: (M, 1024)
    attention_mask: torch.Tensor        # Shape: (M, 1024)
    label: Optional[int] = None
    teacher_prob: Optional[float] = None  # Populated strictly for training documents with teacher logits

def preprocess_doc_tail_chunks(
    doc: LegalJudgmentDoc,
    tokenizer: PreTrainedTokenizerFast,
    chunk_size: int = 1024,
    k_tail: int = 4,
    teacher_prob: Optional[float] = None
) -> PreprocessedDocumentBatch:
    tokens = tokenizer(doc.text, add_special_tokens=False, return_attention_mask=False)["input_ids"]
    
    if len(tokens) == 0:
        tokens = [tokenizer.eos_token_id]

    chunks = [tokens[i : i + chunk_size] for i in range(0, len(tokens), chunk_size)]
    tail_chunks = chunks[-k_tail:]
    M = len(tail_chunks)

    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 3
    input_ids_list = []
    attention_mask_list = []

    for c in tail_chunks:
        pad_len = chunk_size - len(c)
        padded_ids = [pad_id] * pad_len + c
        mask = [0] * pad_len + [1] * len(c)
        input_ids_list.append(padded_ids)
        attention_mask_list.append(mask)

    return PreprocessedDocumentBatch(
        doc_id=doc.doc_id,
        num_chunks=M,
        input_ids=torch.tensor(input_ids_list, dtype=torch.long),
        attention_mask=torch.tensor(attention_mask_list, dtype=torch.long),
        label=doc.label,
        teacher_prob=teacher_prob
    )

class CJPEBatchDataset(Dataset):
    def __init__(
        self,
        docs: List[LegalJudgmentDoc],
        tokenizer: PreTrainedTokenizerFast,
        k_tail: int = 4,
        teacher_targets: Optional[Dict[str, float]] = None
    ):
        self.docs = docs
        self.tokenizer = tokenizer
        self.k_tail = k_tail
        self.teacher_targets = teacher_targets or {}

    def __len__(self) -> int:
        return len(self.docs)

    def __getitem__(self, idx: int) -> PreprocessedDocumentBatch:
        doc = self.docs[idx]
        teacher_prob = self.teacher_targets.get(doc.doc_id, None)
        return preprocess_doc_tail_chunks(
            doc, self.tokenizer, chunk_size=1024, k_tail=self.k_tail, teacher_prob=teacher_prob
        )

print("Constructing preprocessed dataset instances for Phase 4 Distillation...")
# Training dataset receives teacher soft targets; Dev and Test safely receive None
train_dataset = CJPEBatchDataset(train_docs, tokenizer, k_tail=4, teacher_targets=teacher_soft_targets)
dev_dataset = CJPEBatchDataset(dev_docs, tokenizer, k_tail=4, teacher_targets=None)
test_dataset = CJPEBatchDataset(test_docs, tokenizer, k_tail=4, teacher_targets=None)

print(f"Datasets Initialized -> Train: {len(train_dataset):,} (with teacher soft logits), Dev: {len(dev_dataset):,}, Test: {len(test_dataset):,}")


Constructing preprocessed dataset instances for Phase 4 Distillation...
Datasets Initialized -> Train: 37,387 (with teacher soft logits), Dev: 1,988, Test: 1,517


In [5]:
# Cell 5: Upfront Dataloaders & Optimization Step Calibration
BATCH_SIZE = 1           # 1 document per micro-batch (processes up to K=4 chunks of 1024 tokens = 4096 tokens)
GRAD_ACCUM_STEPS = 8     # Effective batch size = 8 documents (smoother gradients, faster local execution)
EPOCHS = 2   # Calibrated for local execution (~1.5 hours on RTX 5050 GPU)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda b: b[0],
    drop_last=False
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda b: b[0]
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda b: b[0]
)

total_optimization_steps = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * EPOCHS
warmup_steps = int(0.10 * total_optimization_steps)

print("=" * 65)
print(f"Step Calibration for Knowledge Distillation Training:")
print(f"  Training Documents:       {len(train_loader):,}")
print(f"  Micro-Batch Size:         {BATCH_SIZE} document (up to 4 x 1024 = 4096 tokens)")
print(f"  Gradient Accumulation:    {GRAD_ACCUM_STEPS} steps")
print(f"  Total Epochs:             {EPOCHS}")
print(f"  Total Optimization Steps: {total_optimization_steps:,}")
print(f"  Warmup Steps (10%):       {warmup_steps:,}")
print("=" * 65)


Step Calibration for Knowledge Distillation Training:
  Training Documents:       37,387
  Micro-Batch Size:         1 document (up to 4 x 1024 = 4096 tokens)
  Gradient Accumulation:    8 steps
  Total Epochs:             2
  Total Optimization Steps: 9,348
  Warmup Steps (10%):       934


In [6]:
# Cell 6: Student Model 3 Backbone Loading from Hugging Face Hub
# Student: Model 3 Statutory Precision Full-MHA Decoder (~109.5M parameters, L=12, d=768, H=12, SwiGLU d_ff=2048)
# Pairs with Teacher Model 4 (153.9M parameters) -> Genuine 28.8% capacity compression

print(f"Downloading Student Model 3 configuration and weights from '{MODEL_REPO}/{BACKBONE_SUBFOLDER}'...")
cfg_path = hf_hub_download(MODEL_REPO, f"{BACKBONE_SUBFOLDER}/config.json", token=HF_TOKEN)
weights_path = hf_hub_download(MODEL_REPO, f"{BACKBONE_SUBFOLDER}/model.safetensors", token=HF_TOKEN)

config = AutoConfig.from_pretrained(cfg_path)
config.use_cache = False
config._name_or_path = MODEL_REPO
base_model = AutoModel.from_config(config)
base_model.config._name_or_path = MODEL_REPO

raw_state_dict = safetensors_load_file(weights_path)
cleaned_sd = {}
for k, v in raw_state_dict.items():
    if k.startswith("model."):
        cleaned_sd[k[len("model."):]] = v
    elif not k.startswith("lm_head."):
        cleaned_sd[k] = v

load_res = base_model.load_state_dict(cleaned_sd, strict=True)
assert len(load_res.missing_keys) == 0, f"Missing keys when loading backbone: {load_res.missing_keys}"
assert len(load_res.unexpected_keys) == 0, f"Unexpected keys when loading backbone: {load_res.unexpected_keys}"
print(f"Base model loaded successfully with strict=True! 100% of Phase 1 weights matched ({len(cleaned_sd)} tensors).")

# Unfreeze 100% of student backbone parameters
for param in base_model.parameters():
    param.requires_grad = True

base_model = base_model.to(device)

total_student_params = sum(p.numel() for p in base_model.parameters())
trainable_student_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)

print("")
print("=" * 65)
print(f"Student Model 3 Backbone Loaded (100% Unfrozen):")
print(f"  Total Student Backbone Parameters:    {total_student_params:,}")
print(f"  Trainable Student Backbone Params:    {trainable_student_params:,} (100.0%)")
print(f"  Teacher Model 4 Backbone Params:      153,889,024")
print(f"  Capacity Compression Ratio:           {(1.0 - total_student_params / 153889024) * 100:.1f}% parameter reduction")
print("=" * 65)
print("")

# Multi-GPU DataParallel Configuration
is_multi_gpu = torch.cuda.is_available() and torch.cuda.device_count() > 1
if is_multi_gpu:
    num_gpus = torch.cuda.device_count()
    print(f"Detected {num_gpus} GPUs! Enabling torch.nn.DataParallel on student backbone...")
    parallel_base_model = nn.DataParallel(base_model)
else:
    parallel_base_model = base_model

Base model loaded successfully with strict=True! 100% of Phase 1 weights matched (110 tensors).

Student Model 3 Backbone Loaded (100% Unfrozen):
  Total Student Backbone Parameters:    109,529,856
  Trainable Student Backbone Params:    109,529,856 (100.0%)
  Teacher Model 4 Backbone Params:      153,889,024
  Capacity Compression Ratio:           28.8% parameter reduction



In [7]:
# Cell 7: Chunk Concat Representation & Learned Attention Pooling
def extract_chunk_concat_representations(hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """
    Computes Concat(MaskedMeanPool, LastTokenHiddenState) for each chunk in the document:
      h_chunk = [MeanPool(H_valid) ; h_last_token] in R^(2 * d_model)
    For Student Model 3 (d_model=768): h_chunk in R^(1536)
    """
    M, seq_len, d_model = hidden_states.shape
    chunk_vectors: List[torch.Tensor] = []

    for m in range(M):
        valid_mask = attention_mask[m].unsqueeze(-1).to(hidden_states.dtype)  # (seq_len, 1)
        valid_tokens = hidden_states[m] * valid_mask                         # (seq_len, d_model)
        
        token_count = valid_mask.sum().clamp(min=1.0)
        mean_pool = valid_tokens.sum(dim=0) / token_count                   # (d_model,)
        last_token = hidden_states[m, -1, :]                                # (d_model,)

        chunk_vectors.append(torch.cat([mean_pool, last_token], dim=-1))    # (2 * d_model,)

    return torch.stack(chunk_vectors, dim=0)                                # (M, 2 * d_model)

class ChunkConcatAttentionPooling(nn.Module):
    """
    Learned scalar attention pooling over M chunk representations:
      u_m = tanh(W_a h_m + b_a)
      alpha_m = softmax(v_a^T u_m)
      h_doc = sum(alpha_m * h_m) in R^(2 * d_model)
    """
    def __init__(self, d_model: int = 768, d_attn: int = 128):
        super().__init__()
        self.in_dim = d_model * 2  # 768 * 2 = 1536
        self.W_a = nn.Linear(self.in_dim, d_attn)
        self.v_a = nn.Linear(d_attn, 1, bias=False)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.W_a.weight)
        nn.init.zeros_(self.W_a.bias)
        nn.init.xavier_uniform_(self.v_a.weight)

    def forward(self, chunk_embeds: torch.Tensor) -> torch.Tensor:
        u = torch.tanh(self.W_a(chunk_embeds))           # (M, d_attn)
        scores = self.v_a(u).squeeze(-1)                 # (M,)
        weights = F.softmax(scores, dim=0)               # (M,)
        doc_embed = torch.sum(weights.unsqueeze(-1) * chunk_embeds, dim=0) # (2 * d_model,)
        return doc_embed

class CourtJudgmentHead(nn.Module):
    """
    2-Layer MLP Classification Head for Court Judgment Prediction:
      LayerNorm(2 * d_model) -> Linear(2 * d_model, hidden_dim) -> GELU -> Dropout(0.1) -> Linear(hidden_dim, 1)
    """
    def __init__(self, d_model: int = 768, hidden_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        self.in_dim = d_model * 2  # 1536
        self.layer_norm = nn.LayerNorm(self.in_dim)
        self.fc1 = nn.Linear(self.in_dim, hidden_dim)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, 1)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, doc_embed: torch.Tensor) -> torch.Tensor:
        if doc_embed.dim() == 1:
            doc_embed = doc_embed.unsqueeze(0)
        h = self.layer_norm(doc_embed)
        h = self.dropout(self.act(self.fc1(h)))
        return self.fc2(h)

attn_pool = ChunkConcatAttentionPooling(d_model=768, d_attn=128).to(device)
head = CourtJudgmentHead(d_model=768, hidden_dim=256, dropout=0.1).to(device)

trainable_pool = sum(p.numel() for p in attn_pool.parameters() if p.requires_grad)
trainable_head = sum(p.numel() for p in head.parameters() if p.requires_grad)
total_trainable = trainable_student_params + trainable_pool + trainable_head

print(f"Initialized ChunkConcatAttentionPooling ({trainable_pool:,} params) and CourtJudgmentHead ({trainable_head:,} params).")
print(f"Total Trainable System Parameters: {total_trainable:,}")


Initialized ChunkConcatAttentionPooling (196,864 params) and CourtJudgmentHead (396,801 params).
Total Trainable System Parameters: 110,123,521


In [8]:
# Cell 8: Combined Distillation Loss Objective & Class Balance
# Loss = 0.4 * BCEWithLogitsLoss(y, z_student) + 0.6 * (T^2) * MSELoss(sigma(z_student / T), p_teacher) with T = 2.0

train_labels = [int(d.label) for d in train_docs if d.label is not None]
pos_count = sum(train_labels)
neg_count = len(train_labels) - pos_count
imbalance_ratio = neg_count / max(1, pos_count)

pos_weight = torch.tensor([imbalance_ratio], dtype=torch.float32, device=device)
hard_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
TEMPERATURE = 2.0
TEMP_SQ = TEMPERATURE ** 2  # 4.0

ALPHA_HARD = 0.4
BETA_KD = 0.6

print(f"Train Set Class Balance: {pos_count} Positive (Allowed), {neg_count} Negative (Dismissed)")
print(f"Hard Loss: BCEWithLogitsLoss (pos_weight: {pos_weight.item():.4f}, weight: {ALPHA_HARD})")
print(f"Distillation Loss: MSELoss (Temperature T={TEMPERATURE}, T^2={TEMP_SQ}, weight: {BETA_KD} * {TEMP_SQ} = {BETA_KD * TEMP_SQ})")


Train Set Class Balance: 15320 Positive (Allowed), 22067 Negative (Dismissed)
Hard Loss: BCEWithLogitsLoss (pos_weight: 1.4404, weight: 0.4)
Distillation Loss: MSELoss (Temperature T=2.0, T^2=4.0, weight: 0.6 * 4.0 = 2.4)


In [9]:
# Cell 9: Differential Optimization Setup & Cosine Schedule
LR_BACKBONE = 5.0e-5   # Student backbone fine-tuning learning rate
LR_HEAD = 1.0e-3       # Attention pooling and Classification Head learning rate

param_groups = [
    {
        "params": [p for p in base_model.parameters() if p.requires_grad],
        "lr": LR_BACKBONE,
        "weight_decay": 0.01,
    },
    {
        "params": list(attn_pool.parameters()) + list(head.parameters()),
        "lr": LR_HEAD,
        "weight_decay": 0.01,
    },
]

optimizer = torch.optim.AdamW(param_groups)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimization_steps
)

print(f"Optimizer: AdamW | Student Backbone LR: {LR_BACKBONE} | Head/Pool LR: {LR_HEAD} | Steps: {total_optimization_steps} (Warmup: {warmup_steps})")


Optimizer: AdamW | Student Backbone LR: 5e-05 | Head/Pool LR: 0.001 | Steps: 9348 (Warmup: 934)


In [10]:
# Cell 10: Evaluation Function & Optimal Decision Threshold Sweeper
def compute_binary_macro_f1(y_true: List[int], y_pred: List[int]) -> float:
    tp = {0: 0, 1: 0}
    fp = {0: 0, 1: 0}
    fn = {0: 0, 1: 0}
    for gt, pr in zip(y_true, y_pred):
        gt, pr = int(gt), int(pr)
        if gt == pr:
            tp[gt] += 1
        else:
            fp[pr] += 1
            fn[gt] += 1
    f1s = []
    for c in [0, 1]:
        p = tp[c] / (tp[c] + fp[c]) if (tp[c] + fp[c]) > 0 else 0.0
        r = tp[c] / (tp[c] + fn[c]) if (tp[c] + fn[c]) > 0 else 0.0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
        f1s.append(f1)
    return sum(f1s) / 2.0

def sweep_decision_threshold(y_true: List[int], y_probs: List[float]) -> Tuple[float, float, Dict[float, float]]:
    best_tau = 0.50
    best_f1 = -1.0
    results: Dict[float, float] = {}
    for tau in np.arange(0.10, 0.91, 0.01):
        tau = round(float(tau), 4)
        preds = [1 if p >= tau else 0 for p in y_probs]
        f1 = compute_binary_macro_f1(y_true, preds)
        results[tau] = f1
        if f1 > best_f1:
            best_f1 = f1
            best_tau = tau
    return best_tau, best_f1, results

def evaluate_dev(model, attn_pool, head, dev_dataset, criterion, device, parallel_model: Optional[nn.Module] = None) -> Tuple[float, float, float, List[int], List[float]]:
    """
    Validates student model on dev set using ground-truth hard labels.
    Safely operates without requiring teacher soft targets.
    Supports Multi-GPU DataParallel when available.
    """
    model.eval()
    attn_pool.eval()
    head.eval()
    total_loss = 0.0
    all_golds: List[int] = []
    all_probs: List[float] = []

    use_dp = parallel_model is not None and torch.cuda.device_count() > 1

    with torch.no_grad():
        for doc_batch in dev_dataset:
            input_ids = doc_batch.input_ids.to(device)
            attention_mask = doc_batch.attention_mask.to(device)
            target = torch.tensor([doc_batch.label], dtype=torch.float32, device=device)

            if use_dp and input_ids.shape[0] >= torch.cuda.device_count():
                outputs = parallel_model(input_ids=input_ids, attention_mask=attention_mask)
            else:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            chunk_reprs = extract_chunk_concat_representations(outputs.last_hidden_state, attention_mask)
            doc_embed = attn_pool(chunk_reprs)
            logit = head(doc_embed).squeeze(-1)

            loss = criterion(logit, target)
            total_loss += loss.item()

            prob = torch.sigmoid(logit).item()
            all_probs.append(prob)
            all_golds.append(int(doc_batch.label))

    avg_loss = total_loss / max(1, len(all_golds))
    best_tau, best_macro_f1, _ = sweep_decision_threshold(all_golds, all_probs)
    return avg_loss, best_macro_f1, best_tau, all_golds, all_probs

In [11]:
# Cell 11: 5-Epoch Knowledge Distillation Training Loop
output_dir = OUTPUT_DIR
os.makedirs(output_dir, exist_ok=True)

best_dev_f1 = 0.0
best_tau = 0.50
history = {"train_loss": [], "bce_loss": [], "kd_loss": [], "dev_loss": [], "dev_macro_f1": []}

use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

print("Commencing {EPOCHS}-epoch Knowledge Distillation training loop for Sub 06 (Student Model 3)...")
start_time = time.time()
global_step = 0

for epoch in range(1, EPOCHS + 1):
    base_model.train()
    attn_pool.train()
    head.train()
    running_total_loss = 0.0
    running_bce = 0.0
    running_kd = 0.0
    optimizer.zero_grad()

    # Shuffle training document indices each epoch
    indices = list(range(len(train_dataset)))
    random.shuffle(indices)

    for step, idx in enumerate(indices):
        doc_batch = train_dataset[idx]
        input_ids = doc_batch.input_ids.to(device)
        attention_mask = doc_batch.attention_mask.to(device)
        target = torch.tensor([doc_batch.label], dtype=torch.float32, device=device)

        with torch.amp.autocast("cuda", enabled=use_amp):
            if is_multi_gpu and input_ids.shape[0] >= torch.cuda.device_count():
                outputs = parallel_base_model(input_ids=input_ids, attention_mask=attention_mask)
            else:
                outputs = base_model(input_ids=input_ids, attention_mask=attention_mask)
            chunk_reprs = extract_chunk_concat_representations(outputs.last_hidden_state, attention_mask)
            doc_embed = attn_pool(chunk_reprs)
            logit = head(doc_embed).squeeze(-1)

            # 1. Hard label classification loss
            bce_loss = hard_criterion(logit, target)

            # 2. Teacher soft target distillation loss
            if doc_batch.teacher_prob is not None:
                p_teacher = torch.tensor([doc_batch.teacher_prob], dtype=torch.float32, device=device)
                p_student_soft = torch.sigmoid(logit / TEMPERATURE)
                kd_loss = F.mse_loss(p_student_soft, p_teacher)
            else:
                kd_loss = torch.tensor(0.0, device=device)

            # 3. Combined loss: 0.4 * BCE + 0.6 * T^2 * MSE
            combined_loss = ALPHA_HARD * bce_loss + (BETA_KD * TEMP_SQ) * kd_loss
            loss_scaled = combined_loss / GRAD_ACCUM_STEPS

        scaler.scale(loss_scaled).backward()
        running_total_loss += combined_loss.item()
        running_bce += bce_loss.item()
        running_kd += kd_loss.item()

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(indices):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in base_model.parameters() if p.requires_grad], 1.0)
            torch.nn.utils.clip_grad_norm_(attn_pool.parameters(), 1.0)
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1

        if (step + 1) % (100 * GRAD_ACCUM_STEPS) == 0:
            avg_tot = running_total_loss / (step + 1)
            avg_bce = running_bce / (step + 1)
            avg_kd = running_kd / (step + 1)
            print(f"  [Epoch {epoch:02d} | Step {step+1:04d}/{len(indices):04d}] Total Loss: {avg_tot:.4f} | Hard BCE: {avg_bce:.4f} | KD MSE: {avg_kd:.4f} | Global Steps: {global_step}")

    epoch_total_loss = running_total_loss / len(indices)
    epoch_bce_loss = running_bce / len(indices)
    epoch_kd_loss = running_kd / len(indices)

    # Validate safely using hard labels on dev set
    dev_loss, dev_f1, tau_star, _, _ = evaluate_dev(base_model, attn_pool, head, dev_dataset, hard_criterion, device, parallel_model=parallel_base_model)

    history["train_loss"].append(epoch_total_loss)
    history["bce_loss"].append(epoch_bce_loss)
    history["kd_loss"].append(epoch_kd_loss)
    history["dev_loss"].append(dev_loss)
    history["dev_macro_f1"].append(dev_f1)

    print(f"Epoch {epoch:02d}/{EPOCHS:02d} Complete | Train Total: {epoch_total_loss:.4f} (BCE: {epoch_bce_loss:.4f}, KD: {epoch_kd_loss:.4f}) | Dev Loss: {dev_loss:.4f} | Dev Macro-F1: {dev_f1:.4f} (tau*: {tau_star:.2f}) | Global Steps: {global_step}")

    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        best_tau = tau_star

        # Save student weights (safetensors and .pt)
        safetensors_save_file(base_model.state_dict(), os.path.join(output_dir, "student_model_weights.safetensors"))
        torch.save(base_model.state_dict(), os.path.join(output_dir, "student_model_weights.pt"))
        torch.save(attn_pool.state_dict(), os.path.join(output_dir, "attn_pool.pt"))
        torch.save(head.state_dict(), os.path.join(output_dir, "head.pt"))
        print(f"  -> NEW BEST Dev Macro-F1: {best_dev_f1:.4f} (tau*: {best_tau:.2f})! Model checkpoint saved.")

total_train_time = time.time() - start_time
print(f"\nTraining completed in {total_train_time / 60:.2f} minutes. Best Dev Macro-F1: {best_dev_f1:.4f} at tau*={best_tau:.2f}")


Commencing {EPOCHS}-epoch Knowledge Distillation training loop for Sub 06 (Student Model 3)...
  [Epoch 01 | Step 0800/37387] Total Loss: 0.3914 | Hard BCE: 0.9055 | KD MSE: 0.0122 | Global Steps: 100
  [Epoch 01 | Step 1600/37387] Total Loss: 0.3809 | Hard BCE: 0.8736 | KD MSE: 0.0131 | Global Steps: 200
  [Epoch 01 | Step 2400/37387] Total Loss: 0.3676 | Hard BCE: 0.8438 | KD MSE: 0.0125 | Global Steps: 300
  [Epoch 01 | Step 3200/37387] Total Loss: 0.3600 | Hard BCE: 0.8341 | KD MSE: 0.0110 | Global Steps: 400
  [Epoch 01 | Step 4000/37387] Total Loss: 0.3521 | Hard BCE: 0.8198 | KD MSE: 0.0101 | Global Steps: 500
  [Epoch 01 | Step 4800/37387] Total Loss: 0.3462 | Hard BCE: 0.8088 | KD MSE: 0.0095 | Global Steps: 600
  [Epoch 01 | Step 5600/37387] Total Loss: 0.3427 | Hard BCE: 0.8028 | KD MSE: 0.0090 | Global Steps: 700
  [Epoch 01 | Step 6400/37387] Total Loss: 0.3402 | Hard BCE: 0.7996 | KD MSE: 0.0085 | Global Steps: 800
  [Epoch 01 | Step 7200/37387] Total Loss: 0.3378 | Hard 

In [12]:
# Cell 12: Test Inference, Calibrated Threshold Prediction & Codabench Packaging
# 1. Load Best Saved Checkpoint Weights
weights_path = os.path.join(output_dir, "student_model_weights.safetensors")
pool_path = os.path.join(output_dir, "attn_pool.pt")
head_path = os.path.join(output_dir, "head.pt")

if os.path.exists(weights_path):
    print("Loading best Student Model checkpoint for final evaluation...")
    base_model.load_state_dict(safetensors_load_file(weights_path), strict=True)
    attn_pool.load_state_dict(torch.load(pool_path, map_location=device))
    head.load_state_dict(torch.load(head_path, map_location=device))

# 2. Final Evaluation on Dev Split to confirm optimal tau*
print("Executing final dev evaluation sweep...")
final_dev_loss, final_dev_f1, optimal_tau, dev_golds, dev_probs = evaluate_dev(
    base_model, attn_pool, head, dev_dataset, hard_criterion, device, parallel_model=parallel_base_model
)
print(f"Final Dev Macro-F1: {final_dev_f1:.4f} at Optimal Decision Threshold tau* = {optimal_tau:.2f}")

np.save(os.path.join(output_dir, "dev_probs.npy"), np.array(dev_probs, dtype=np.float32))

# 3. Predict on Test Split
base_model.eval()
attn_pool.eval()
head.eval()
test_probs: List[float] = []

print(f"Generating test predictions across {len(test_dataset):,} test documents...")
with torch.no_grad():
    for doc_batch in test_dataset:
        input_ids = doc_batch.input_ids.to(device)
        attention_mask = doc_batch.attention_mask.to(device)

        if is_multi_gpu and input_ids.shape[0] >= torch.cuda.device_count():
            outputs = parallel_base_model(input_ids=input_ids, attention_mask=attention_mask)
        else:
            outputs = base_model(input_ids=input_ids, attention_mask=attention_mask)
        chunk_reprs = extract_chunk_concat_representations(outputs.last_hidden_state, attention_mask)
        doc_embed = attn_pool(chunk_reprs)
        logit = head(doc_embed).squeeze(-1)

        prob = torch.sigmoid(logit).item()
        test_probs.append(prob)

sub06_test_probs_path = os.path.join(output_dir, "sub06_test_probs.npy")
np.save(sub06_test_probs_path, np.array(test_probs, dtype=np.float32))
print(f"Exported test probabilities to '{sub06_test_probs_path}' ({len(test_probs)} items).")

# 4. Generate Codabench Formatted prediction.json using Optimal Threshold
prediction_list = []
for doc, p in zip(test_docs, test_probs):
    pred_label = 1 if p >= optimal_tau else 0
    prediction_list.append({"id": doc.doc_id, "pred": int(pred_label)})

pred_path = os.path.join(output_dir, "prediction.json")
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(prediction_list, f, indent=2)

print(f"Exported '{pred_path}' with {len(prediction_list)} predictions (Threshold tau* = {optimal_tau:.2f}).")

# 5. Export Performance Metrics and Evaluation Report
metrics = {
    "submission_id": "sub06_model3_distill",
    "student_model_backbone": "Model 3 Statutory Precision Full-MHA Decoder (109.5M)",
    "teacher_model_backbone": "Model 4 Peak Capacity Boundary Decoder (153.9M)",
    "adaptation_method": "Knowledge Distillation (BCE + MSE Temperature Scaled Soft Logits)",
    "temperature": float(TEMPERATURE),
    "capacity_reduction_percent": 28.8,
    "best_dev_macro_f1": float(final_dev_f1),
    "optimal_decision_threshold": float(optimal_tau),
    "dev_loss": float(final_dev_loss),
    "total_optimization_steps": int(total_optimization_steps),
    "epochs_trained": int(EPOCHS),
}
with open(os.path.join(output_dir, "val_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

report = {
    "metrics": metrics,
    "training_history": history,
}
with open(os.path.join(output_dir, "training_evaluation_report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

# 6. Plot Loss Curve & Macro-F1 Progression
plt.figure(figsize=(14, 4))
plt.subplot(1, 3, 1)
plt.plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], label="Total Distill Loss", marker="o")
plt.plot(range(1, len(history["dev_loss"]) + 1), history["dev_loss"], label="Dev Loss", marker="s")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Sub 06: Total Distillation Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(range(1, len(history["bce_loss"]) + 1), history["bce_loss"], label="Hard BCE", marker="o", color="blue")
plt.plot(range(1, len(history["kd_loss"]) + 1), history["kd_loss"], label="Soft KD MSE", marker="x", color="purple")
plt.xlabel("Epoch")
plt.ylabel("Loss Component")
plt.title("Loss Components (Hard BCE vs Soft KD)")
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(range(1, len(history["dev_macro_f1"]) + 1), history["dev_macro_f1"], label="Dev Macro-F1", marker="^", color="green")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title(f"Dev Macro-F1 (Peak: {final_dev_f1:.4f} at tau*={optimal_tau:.2f})")
plt.legend()
plt.grid(True)

plt.tight_layout()
plot_path = os.path.join(output_dir, "loss_plot.png")
plt.savefig(plot_path, dpi=150)
plt.close()
print(f"Saved training visualization to '{plot_path}'.")

# 7. Strict Codabench Packaging: Package prediction.json directly at the zip root
zip_path = os.path.join(output_dir, "submission.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, arcname="prediction.json")

# Integrity Verification of submission.zip
with zipfile.ZipFile(zip_path, "r") as zf:
    infolist = zf.infolist()
    assert len(infolist) == 1, f"Expected exactly 1 file in submission.zip, found {len(infolist)}"
    assert infolist[0].filename == "prediction.json", f"Found '{infolist[0].filename}' instead of 'prediction.json' at root"

assert len(prediction_list) == len(test_docs), f"Test doc count mismatch: {len(prediction_list)} vs {len(test_docs)}"
for item in prediction_list:
    assert item["pred"] in (0, 1), f"Invalid prediction: {item['pred']}"

print(f"Codabench submission successfully verified and packaged: '{zip_path}' ({os.path.getsize(zip_path):,} bytes, {len(prediction_list)} test cases).")

Loading best Student Model checkpoint for final evaluation...
Executing final dev evaluation sweep...
Final Dev Macro-F1: 0.6394 at Optimal Decision Threshold tau* = 0.46
Generating test predictions across 1,517 test documents...
Exported test probabilities to 'checkpoint_sub06\sub06_test_probs.npy' (1517 items).
Exported 'checkpoint_sub06\prediction.json' with 1517 predictions (Threshold tau* = 0.46).
Saved training visualization to 'checkpoint_sub06\loss_plot.png'.
Codabench submission successfully verified and packaged: 'checkpoint_sub06\submission.zip' (6,540 bytes, 1517 test cases).


In [22]:
# Cell 13: Local Artifact Verification & Atomic Hugging Face Hub Upload (with Executed Notebook Sync)
notebook_filename = "sub06_model3_distill.ipynb"
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"  # Disables xet and prevents 403 Forbidden

api = HfApi(token=HF_TOKEN)
api.upload_folder(
    folder_path=output_dir,
    repo_id=MODEL_REPO,
    path_in_repo=TARGET_SUBFOLDER,
    commit_message=f"Upload Phase 4 Sub 06 (Model 3 Knowledge Distillation) - Dev F1: {final_dev_f1:.4f}",
    ignore_patterns=["*.pt"]  # Excludes duplicate uncompressed .pt weights
)

# 1. Save Tokenizer and Model Configuration
tokenizer.save_pretrained(output_dir)
config.save_pretrained(output_dir)

# 2. Stage/Preserve the Executed Notebook File into output_dir
def stage_executed_notebook(nb_name: str, dest_dir: str):
    search_paths = [
        nb_name,
        os.path.join(os.getcwd(), nb_name),
        os.path.join("/kaggle/working", nb_name),
        os.path.join("/kaggle/working", os.path.basename(nb_name)),
        "/kaggle/working/.virtual_documents/__notebook_source__.ipynb",
        "/kaggle/working/__notebook_source__.ipynb",
    ]
    staged = False
    for p in search_paths:
        if os.path.exists(p) and os.path.isfile(p):
            dest_file = os.path.join(dest_dir, nb_name)
            shutil.copy2(p, dest_file)
            print(f"Successfully staged executed notebook '{p}' -> '{dest_file}' ({os.path.getsize(dest_file):,} bytes).")
            staged = True
            break
    if not staged:
        try:
            import IPython
            IPython.display.display(IPython.display.Javascript('IPython.notebook.save_notebook()'))
            time.sleep(2)
        except Exception:
            pass
        for p in search_paths:
            if os.path.exists(p) and os.path.isfile(p):
                dest_file = os.path.join(dest_dir, nb_name)
                shutil.copy2(p, dest_file)
                print(f"Successfully staged executed notebook '{p}' -> '{dest_file}'.")
                staged = True
                break
    if not staged:
        print(f"Notice: '{nb_name}' not found locally. Ensure notebook is saved before final upload.")

stage_executed_notebook(notebook_filename, output_dir)

# 3. Write Clean YAML README.md for Hugging Face Hub Compliance
readme_path = os.path.join(output_dir, "README.md")
readme_content = f"""---
base_model: {MODEL_REPO}
tags:
- legal
- legal-nlp
- court-judgment-prediction
- ildc
- knowledge-distillation
- student-model
---

# Phase 4 Submission 06: Model 3 Statutory Precision Full-MHA (Knowledge Distillation Student)
- **Base Student Model:** `{MODEL_REPO}` (subfolder: `{BACKBONE_SUBFOLDER}`)
- **Teacher Model:** Model 4 Peak Capacity Boundary Decoder (153.9M, Sub 01 Anchor)
- **Task:** Court Judgment Prediction (CJPE - Binary Classification)
- **Role:** Compulsory Knowledge Distillation (validated capacity reduction: 153.9M -> 109.5M, 28.8% reduction)
- **Adaptation Method:** Full Parameter Student Training with 0.4 * BCE + 0.6 * T^2 * MSE distillation loss (T=2.0)
- **Best Dev Macro-F1:** {final_dev_f1:.4f} (Optimal Decision Threshold tau*: {optimal_tau:.2f})
- **Trainable Parameters:** {total_trainable:,} (100.0% of student)
- **Test Predictions:** Available in `prediction.json` and packaged in `submission.zip`
"""
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_content)

# 4. Verify All Required Artifacts Exist Locally Before Upload
required_files = [
    "student_model_weights.safetensors",
    "attn_pool.pt",
    "head.pt",
    "config.json",
    "tokenizer.json",
    "val_metrics.json",
    "training_evaluation_report.json",
    "loss_plot.png",
    "dev_probs.npy",
    "sub06_test_probs.npy",
    "prediction.json",
    "submission.zip",
    "README.md"
]

missing_files = [f for f in required_files if not os.path.exists(os.path.join(output_dir, f))]
assert not missing_files, f"Cannot proceed with upload! Missing local files: {missing_files}"
print(f"All required local artifacts verified present in '{output_dir}'!")

# 5. Atomic Single-Commit Folder Upload to Hugging Face Hub
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Enter Hugging Face Write Token: ")

api = HfApi(token=HF_TOKEN)
print(f"Initiating atomic upload to '{MODEL_REPO}' under '{TARGET_SUBFOLDER}'...")
api.upload_folder(
    folder_path=output_dir,
    repo_id=MODEL_REPO,
    path_in_repo=TARGET_SUBFOLDER,
    commit_message=f"Upload Phase 4 Sub 06 (Model 3 KD Student) - Dev F1: {final_dev_f1:.4f}",
    token=HF_TOKEN
)
print(f"Upload complete! Artifacts & executed notebook published to: https://huggingface.co/{MODEL_REPO}/tree/main/{TARGET_SUBFOLDER}")


ConnectionError: Network error: Request error: HTTP status client error (403 Forbidden), domain: https://huggingface.co/api/models/sarweshbarnwal/CS781-Legal-Models/xet-write-token/main

In [ ]:
import os
import sys
import shutil
import time
import json
import re
import huggingface_hub.constants as constants
import huggingface_hub.hf_api as hf_api
from huggingface_hub import HfApi

# 1. Permanently bypass hf_xet in active Python memory (prevents 403 Forbidden)
constants.HF_HUB_DISABLE_XET = True
hf_api.is_xet_available = lambda: False

# 2. Configuration & Credentials
MODEL_REPO = "sarweshbarnwal/CS781-Legal-Models"
TARGET_SUBFOLDER = "Phase_4/sub06_model3_distill"
output_dir = "checkpoint_sub06" if os.path.exists("checkpoint_sub06") else os.path.join(os.getcwd(), "checkpoint_sub06")
HF_TOKEN = "REDACTED_HF_TOKEN"

api = HfApi(token=HF_TOKEN)

# 3. Stage the executed notebook and REDACT secret tokens (passes HF secret scanner)
notebook_name = "sub06_model3_distill.ipynb"
for src_candidate in [notebook_name, f"../{notebook_name}", os.path.join(os.getcwd(), notebook_name)]:
    if os.path.exists(src_candidate) and os.path.isfile(src_candidate):
        dest_path = os.path.join(output_dir, notebook_name)
        with open(src_candidate, "r", encoding="utf-8") as f:
            nb_text = f.read()
        # Redact raw HF tokens to satisfy Hugging Face secret scanner
        nb_text_clean = re.sub(r"hf_[a-zA-Z0-9]{30,}", "REDACTED_HF_TOKEN", nb_text)
        with open(dest_path, "w", encoding="utf-8") as f:
            f.write(nb_text_clean)
        print(f" Staged and sanitized notebook -> '{dest_path}'")
        break

# 4. Push all deliverables, metrics, reports, plots, and submission.zip
print(f" Pushing deliverables and submission to Hugging Face Hub ('{TARGET_SUBFOLDER}')...")
commit_url = api.upload_folder(
    folder_path=output_dir,
    repo_id=MODEL_REPO,
    path_in_repo=TARGET_SUBFOLDER,
    commit_message="Upload Phase 4 Sub 06: KD Deliverables & Sanitized Notebook",
    ignore_patterns=["student_model_weights.pt", "student_model_weights.safetensors"],  # Uploads all deliverables instantly
    token=HF_TOKEN,
)
print(" Success! All deliverables and submission.zip published to:")
print(f"   https://huggingface.co/{MODEL_REPO}/tree/main/{TARGET_SUBFOLDER}")

 Staged notebook 'sub06_model3_distill.ipynb' -> 'checkpoint_sub06\sub06_model3_distill.ipynb'
 Pushing deliverables and submission to Hugging Face Hub ('Phase_4/sub06_model3_distill')...


Upload 0 LFS files: 0it [00:00, ?it/s]


BadRequestError: (Request ID: Root=1-6ad9366d-6f27dc616696c4213fca9729;acdeb326-a1c1-4677-ac3c-30ce2aacc932)

Bad request for commit endpoint:
It appears that one or more of your files contain valid Hugging Face secrets, such as tokens or API keys. It is important to keep your secrets private and not expose them in code that is publicly accessible. Read more: https://huggingface.co/docs/hub/spaces-overview#managing-secrets Offending files: - Phase_4/sub06_model3_distill/sub06_model3_distill.ipynb (ref: refs/heads/main, token: 'REDACTED_HF_TOKEN')
It appears that one or more of your files contain valid Hugging Face secrets, such as tokens or API keys. It is important to keep your secrets private and not expose them in code that is publicly accessible. Read more: https://huggingface.co/docs/hub/spaces-overview#managing-secrets  Offending files:   - Phase_4/sub06_model3_distill/sub06_model3_distill.ipynb (ref: refs/heads/main, token: 'REDACTED_HF_TOKEN')